In [2]:
import json


In [3]:
with open("bio_tags.json","r") as f:
    bio_data = json.load(f)
    

In [4]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict, ClassLabel
from torch.utils.data import DataLoader

# 1. Load and preprocess the data
def preprocess_data_in_chunks(biotagged_data, tokenizer, max_length=128):
    """
    Preprocesses the biotagged data in manageable chunks, ensuring labels align with tokens.
    
    Args:
        biotagged_data (list of list): List of token-tag pairs.
        tokenizer (transformers tokenizer): Tokenizer to be used.
        max_length (int): Max token length for the model.
    
    Returns:
        dict: Encodings with input_ids, attention_masks, and aligned labels.
    """
    chunks = []
    current_chunk_tokens = []
    current_chunk_labels = []

    # Helper function to finalize and store a chunk
    def finalize_chunk():
        if current_chunk_tokens:
            chunks.append((current_chunk_tokens, current_chunk_labels))
    
    # Iterate over each token-tag pair
    for token, tag in biotagged_data:
        current_chunk_tokens.append(token)
        current_chunk_labels.append(tag)

        # If the current chunk exceeds max_length (minus 2 for [CLS] and [SEP] tokens), finalize it
        if len(current_chunk_tokens) >= max_length - 2:
            finalize_chunk()
            current_chunk_tokens = []
            current_chunk_labels = []

    # Finalize the last chunk
    finalize_chunk()

    # Tokenize and align labels for each chunk
    encodings = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    label2id = {
        "B-LINK": 0,
        "I-LINK": 1,
        "O": 2,
    }

    for tokens, labels in chunks:
        # Tokenize the chunk
        tokenized = tokenizer(tokens, padding="max_length", truncation=True, max_length=max_length, is_split_into_words=True)
        word_ids = tokenized.word_ids()

        # Align labels with word_ids
        aligned_labels = []
        for word_id in word_ids:
            if word_id is None:  # Special tokens like [CLS] and [SEP]
                aligned_labels.append(-100)  # Ignore index for the loss function
            else:
                aligned_labels.append(label2id[labels[word_id]])

        encodings["input_ids"].append(tokenized["input_ids"])
        encodings["attention_mask"].append(tokenized["attention_mask"])
        encodings["labels"].append(aligned_labels)

    return encodings








In [5]:
# 2. Load RuBERT and tokenizer
tokenizer = BertTokenizerFast.from_pretrained("DeepPavlov/rubert-base-cased")
model = BertForTokenClassification.from_pretrained("DeepPavlov/rubert-base-cased", num_labels=3)


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

# 3. Prepare your biotagged data
# For example, using a list of tokenized sentences: [["token1", "B-Title"], ["token2", "I-Title"], ...]
# For your case, replace `biotagged_data` with your actual dataset.
""
# Example biotagged data (replace with your actual data)
biotagged_data = bio_data

# Preprocess the data
encodings = preprocess_data_in_chunks(biotagged_data, tokenizer, max_length=128)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [7]:

# 4. Split the data into training and validation sets
dataset = Dataset.from_dict(encodings)
train_test_split = dataset.train_test_split(test_size=0.1)

In [8]:
# 5. Convert into DatasetDict
dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

In [9]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=5,              # number of training epochs
    per_device_train_batch_size=128,   # batch size for training
    per_device_eval_batch_size=128,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="epoch",     # evaluate every epoch
    save_strategy="epoch",           # save every epoch
    load_best_model_at_end=True ,     # load the best model when finished training
    report_to="none",
    learning_rate = 5e-5,
    logging_strategy="no"
)

trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=dataset_dict['train'], # training dataset
    eval_dataset=dataset_dict['test'],   # evaluation dataset
    tokenizer=tokenizer,                 # tokenizer for tokenization during inference
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_23/3595352152.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:

# 7. Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.010754
2,No log,0.009456
3,No log,0.007987
4,No log,0.008205
5,No log,0.009051


TrainOutput(global_step=3690, training_loss=0.015081955617682398, metrics={'train_runtime': 5388.365, 'train_samples_per_second': 87.614, 'train_steps_per_second': 0.685, 'total_flos': 3.083950190835072e+16, 'train_loss': 0.015081955617682398, 'epoch': 5.0})

In [11]:

# 9. Save the trained model

model.save_pretrained("/kaggle/working/rubert_ner_model")
tokenizer.save_pretrained("/kaggle/working/rubert_ner_model")

('/kaggle/working/rubert_ner_model/tokenizer_config.json',
 '/kaggle/working/rubert_ner_model/special_tokens_map.json',
 '/kaggle/working/rubert_ner_model/vocab.txt',
 '/kaggle/working/rubert_ner_model/added_tokens.json',
 '/kaggle/working/rubert_ner_model/tokenizer.json')

In [12]:
import shutil

# Define the directory containing the model and the output zip file
model_directory = "/kaggle/working/rubert_ner_model"
output_zip_file = "/kaggle/working/rubert_ner_model.zip"

# Create a zip archive
shutil.make_archive(output_zip_file.replace('.zip', ''), 'zip', model_directory)

print(f"Model zipped to {output_zip_file}")

Model zipped to /kaggle/working/rubert_ner_model.zip


In [13]:

# 8. Evaluate the model
trainer.evaluate()


{'eval_loss': 0.007986572571098804,
 'eval_runtime': 37.6267,
 'eval_samples_per_second': 278.844,
 'eval_steps_per_second': 2.179,
 'epoch': 5.0}

In [14]:
def predict_entities(text):
    # Split the text into words (pre-tokenization)
    words = text.split()

    # Tokenize the words list using the tokenizer (now it will process the words correctly)
    inputs = tokenizer(words, return_tensors="pt", padding=True, truncation=True, is_split_into_words=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    predictions = torch.argmax(logits, dim=-1)

    # Align predictions with words
    word_ids = inputs.word_ids()  # This tells us which token corresponds to which word
    result = []

    previous_word_idx = None
    for word_id, label_id in zip(word_ids, predictions[0]):
        if word_id is None:  # Skip special tokens
            continue
        if word_id != previous_word_idx:  # New word
            word = words[word_id]
            label = model.config.id2label[label_id.item()]
            result.append((word, label))
        previous_word_idx = word_id

    return result


s)